Write Python code and collaborate in real time. Your code runs in Modal's
**serverless cloud**, and anyone in the same workspace can join.

This notebook comes with some common Python libraries installed. Run
cells with `Shift+Enter`.

In [11]:
import sys
sys.executable

'/home/ubuntu/workspace/modded-nanogpt/.venv/bin/python'

In [12]:
%cd /home/ubuntu/workspace/modded-nanogpt/

/home/ubuntu/workspace/modded-nanogpt


In [13]:
# Setup fast transfer libraries
%env HF_TRANSFER=1
%env HF_HUB_ENABLE_HF_TRANSFER=1
%env MODDED_NANOGPT_CACHE=/home/ubuntu/workspace/modded-nanogpt/data/

env: HF_TRANSFER=1
env: HF_HUB_ENABLE_HF_TRANSFER=1
env: MODDED_NANOGPT_CACHE=/home/ubuntu/workspace/modded-nanogpt/data/


In [14]:
# download fine web
import os

from huggingface_hub import hf_hub_download
from concurrent.futures import ThreadPoolExecutor, as_completed

def get(fname):
    local_dir = os.path.join(os.environ['MODDED_NANOGPT_CACHE'], 'fineweb10B')
    if not os.path.exists(os.path.join(local_dir, fname)):
        hf_hub_download(
            repo_id="kjj0/fineweb10B-gpt2",
            filename=fname,
            repo_type="dataset",
            local_dir=local_dir
        )

num_chunks = 8  # full fineweb10B use 103. Each chunk is 100M tokens

files = ["fineweb_val_%06d.bin" % 0] + [
    "fineweb_train_%06d.bin" % i for i in range(1, num_chunks + 1)
]

with ThreadPoolExecutor(max_workers=8) as executor:  # adjust workers
    futures = {executor.submit(get, f): f for f in files}
    for future in as_completed(futures):
        fname = futures[future]
        try:
            future.result()
            print(f"Downloaded {fname}")
        except Exception as e:
            print(f"Failed {fname}: {e}")

Downloaded fineweb_train_000001.bin
Downloaded fineweb_train_000002.bin
Downloaded fineweb_val_000000.bin
Downloaded fineweb_train_000005.bin
Downloaded fineweb_train_000004.bin
Downloaded fineweb_train_000003.bin
Downloaded fineweb_train_000006.bin
Downloaded fineweb_train_000007.bin
Downloaded fineweb_train_000008.bin


In [15]:
# print all the version for the following libraries: pytorc, cuda
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA version: {torch.version.cuda}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"Current CUDA device: {torch.cuda.current_device()}")
    print(f"CUDA device name: {torch.cuda.get_device_name()}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}")
    print(f"SMs: {props.multi_processor_count}")
else:
    print("WARNING NO VALID CUDA SETUP FOUND")

PyTorch version: 2.8.0+cu128
CUDA version: 12.8
CUDA available: True
CUDA device count: 8
Current CUDA device: 0
CUDA device name: NVIDIA H100 80GB HBM3
GPU: NVIDIA H100 80GB HBM3
SMs: 132


In [19]:
import multiprocessing

max(multiprocessing.cpu_count() - 1, 1) // 8.


25.0

In [ ]:
# !torchrun --standalone --nproc_per_node={torch.cuda.device_count()} train_gpt.py

W0923 22:20:29.866000 23015 torch/distributed/run.py:774] 
W0923 22:20:29.866000 23015 torch/distributed/run.py:774] *****************************************
W0923 22:20:29.866000 23015 torch/distributed/run.py:774] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0923 22:20:29.866000 23015 torch/distributed/run.py:774] *****************************************
W0923 22:20:33.087000 23015 torch/distributed/elastic/agent/server/api.py:723] Received 2 death signal, shutting down workers
W0923 22:20:33.088000 23015 torch/distributed/elastic/multiprocessing/api.py:900] Sending process 23084 closing signal SIGINT
W0923 22:20:33.088000 23015 torch/distributed/elastic/multiprocessing/api.py:900] Sending process 23085 closing signal SIGINT
W0923 22:20:33.088000 23015 torch/distributed/elastic/multiprocessing/api.py:900] Sending proce